## Import Region data into MongoDB
The goal is to import the following location (region) data into our database:

- Countries and Continents (with geometry and coordinates)
- States and Districts (with geometry and coordinates, postal code, address region)
- Cities, localities and neighbourhoods (with geometry, postal code, address region, address locality, street address)

Having this data locally makes it cheaper and reduces latency for us.

#### Install Dependencies

In [1]:
!uv sync --python 3.13

zsh:1: command not found: uv


#### Import Libraries

In [2]:
import geopandas as gpd
import pandas as pd
import math
import unicodedata
import difflib
from typing import Optional
import numpy as np
import os
import httpx
import zipfile
import tenacity
import pathlib
from concurrent.futures import ProcessPoolExecutor

### Data Extraction Helpers

Let's define helper functions to help us download and cache datasets for processing.

In [3]:

DATA_DIR = "data"

@tenacity.retry(
    stop=tenacity.stop_after_attempt(5),            # retry up to 5 times
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=20),  # exponential backoff
    reraise=True
)
async def fetch_file(url: str, filepath: str):
    """Fetch a file asynchronously with retries."""
    timeout = httpx.Timeout(30.0, connect=10.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.get(url)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
    return filepath


async def download_and_extract(url: str, extract_to: str = DATA_DIR, force: bool = False, filename: str | None = None) -> list[str]:
    """
    Async download and extract a file (zip or direct) from a URL.

    Retries handled by tenacity.
    """
    os.makedirs(extract_to, exist_ok=True)
    filename = filename or os.path.basename(url)
    filepath = os.path.join(extract_to, filename)

    if not os.path.exists(filepath) or force:
        print(f"Downloading from {url}...")
        await fetch_file(url, filepath)
        print("Download complete.")
    else:
        print(f"File already exists: {filepath}, skipping download.")

    extracted_files = []
    if zipfile.is_zipfile(filepath):
        with zipfile.ZipFile(filepath, "r") as z:
            z.extractall(extract_to)
            extracted_files = [os.path.join(extract_to, name) for name in z.namelist()]
        print("Extraction complete.")
    else:
        extracted_files = [filepath]

    return extracted_files

### Extract Continent Data

Let's start from the top- extract continent data first

In [4]:
def build_continent_aliases(world: gpd.GeoDataFrame):
    """Build a flat list of alternative names for each continent."""

    continent_altnames = {}
    for continent, group in world.groupby("CONTINENT"):
        continent_altnames[continent] = []
    return continent_altnames


async def load_continents():
    """Load continent geometries with name, centroid coordinates, and alternative names."""
    extracted_files = await download_and_extract(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    # Load countries
    world = gpd.read_file(shapefile)
    
    print("Continent data sample:")
    display(world.head())

    # Dissolve into continents
    continents = world.dissolve(by="CONTINENT").reset_index()[["CONTINENT", "geometry"]]

    # Reproject to a projected CRS before centroid calculation
    continents_proj = continents.to_crs("ESRI:54034")  # Equal Earth projection
    centroids_proj = continents_proj.centroid

    # Convert centroids back to lat/lon (WGS84)
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")

    # Assign coordinates as [lat, lon]
    continents["coordinates"] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Build alt names
    continent_altnames = build_continent_aliases(world)
    continents["aliases"] = continents["CONTINENT"].map(continent_altnames)
    
    continents["level"] = "continent"

    # Rename column for clarity
    continents = continents.rename(columns={"CONTINENT": "name"})

    return continents

continents = await load_continents()
continents

File already exists: data/ne_110m_admin_0_countries.zip, skipping download.
Extraction complete.
Continent data sample:


,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,None,None,None,None,None,None,None,None,None,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,None,None,Unrecognized,None,None,None,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


,name,geometry,coordinates,aliases,level
0,Africa,"MULTIPOLYGON (((-11.43878 6.78592, -11.70819 6...","[6.032181157557932, 18.40040406024284]",[],continent
1,Antarctica,"MULTIPOLYGON (((-61.13898 -79.98137, -60.61012...","[-75.50522868903988, 35.64346078762372]",[],continent
2,Asia,"MULTIPOLYGON (((48.67923 14.0032, 48.23895 13....","[29.888226373617375, 84.71223389814556]",[],continent
3,Europe,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2....","[56.65066091381333, 73.8348995832024]",[],continent
4,North America,"MULTIPOLYGON (((-155.22217 19.23972, -155.5421...","[47.14289012790487, -96.36461725674786]",[],continent
5,Oceania,"MULTIPOLYGON (((147.91405 -43.21152, 147.56456...","[-24.573805155566546, 136.50696195370307]",[],continent
6,Seven seas (open ocean),"POLYGON ((68.935 -48.625, 69.58 -48.94, 70.525...","[-49.30442568487861, 69.53123972063625]",[],continent
7,South America,"MULTIPOLYGON (((-68.63999 -55.58002, -69.2321 ...","[-13.587586113083029, -60.64773056426989]",[],continent


In [5]:
# Drop Seven Seas
# note: Oceania is taken as a continent, and not Australia here- which is referred to as a country.
continents = continents[continents["name"] != "Seven seas (open ocean)"]
continents = continents.reset_index(drop=True)
continents

,name,geometry,coordinates,aliases,level
0,Africa,"MULTIPOLYGON (((-11.43878 6.78592, -11.70819 6...","[6.032181157557932, 18.40040406024284]",[],continent
1,Antarctica,"MULTIPOLYGON (((-61.13898 -79.98137, -60.61012...","[-75.50522868903988, 35.64346078762372]",[],continent
2,Asia,"MULTIPOLYGON (((48.67923 14.0032, 48.23895 13....","[29.888226373617375, 84.71223389814556]",[],continent
3,Europe,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2....","[56.65066091381333, 73.8348995832024]",[],continent
4,North America,"MULTIPOLYGON (((-155.22217 19.23972, -155.5421...","[47.14289012790487, -96.36461725674786]",[],continent
5,Oceania,"MULTIPOLYGON (((147.91405 -43.21152, 147.56456...","[-24.573805155566546, 136.50696195370307]",[],continent
6,South America,"MULTIPOLYGON (((-68.63999 -55.58002, -69.2321 ...","[-13.587586113083029, -60.64773056426989]",[],continent


### Extract Country Data
Next, let's import all the countries in the continents we have extracted so far.

In [6]:
def build_country_aliases(countries: gpd.GeoDataFrame):
    """Build a flat list of alternative names for each country."""
    alt_name_columns = [
        col for col in countries.columns
        if col.startswith("NAME_") or col in ["NAME_ALT", "NAME_LONG", "FORMAL_EN", "FORMAL_FR"]
    ]

    country_altnames = {}
    for _, row in countries.iterrows():
        names = set()
        for col in alt_name_columns:
            v = row[col]
            if v not in [None, "", -99] and isinstance(v, str) and v != row["ADMIN"]:
                names.add(str(v))  # ensure all names are strings
        country_altnames[row["ADMIN"]] = sorted(names)

    return country_altnames


async def load_countries():
    """Load all countries with name, centroid coordinates, and alternative names."""
    extracted_files = await download_and_extract(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    # Load countries
    countries = gpd.read_file(shapefile)
    
    print("Country data sample:")
    display(countries.head())

    # Build aliases before dropping columns
    country_altnames = build_country_aliases(countries)

    # Keep only useful columns
    countries = countries[['ADMIN', 'ISO_A2', 'geometry']]

    # Project to a suitable projected CRS for accurate centroid computation
    countries_proj = countries.to_crs("ESRI:54034")  # Equal Earth
    centroids_proj = countries_proj.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")

    # Assign coordinates as [lat, lon]
    countries['coordinates'] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Map aliases
    countries['aliases'] = countries['ADMIN'].map(country_altnames)

    # Optionally rename columns for clarity
    countries = countries.rename(columns={"ADMIN": "name", "ISO_A2": "iso_a2"})
    
    countries["level"] = "country"

    return countries

countries = await load_countries()
countries

File already exists: data/ne_110m_admin_0_countries.zip, skipping download.
Extraction complete.
Country data sample:


,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,None,None,None,None,None,None,None,None,None,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,None,None,Unrecognized,None,None,None,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


,name,iso_a2,geometry,coordinates,aliases,level
0,Fiji,FJ,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...","[-17.31277520899394, 163.77640725005324]","[Fidji, Fidschi, Fidzsi-szigetek, Fidżi, Figi,...",country
1,United Republic of Tanzania,TZ,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...","[-6.237730725476361, 34.74766857623911]","[Tansania, Tanzania, Tanzanie, Tanzanya, Tanzá...",country
2,Western Sahara,EH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...","[24.24928060599045, -12.162183576752351]","[Batı Sahra, Nyugat-Szahara, Sahara Barat, Sah...",country
3,Canada,CA,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...","[57.62966510888876, -97.64486003045715]","[Canadá, Kanada, Καναδάς, Канада, קנדה, كندا, ...",country
4,United States of America,US,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...","[42.075007554015286, -107.5447286816333]","[Amerika Birleşik Devletleri, Amerika Serikat,...",country
...,...,...,...,...,...,...
172,Republic of Serbia,RS,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...","[44.21138805382481, 20.828267375143835]","[Serbia, Serbie, Serbien, Servië, Szerbia, Sér...",country
173,Montenegro,ME,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...","[42.78633660560693, 19.285947508559723]","[Czarnogóra, Karadağ, Montenegró, Monténégro, ...",country
174,Kosovo,-99,"POLYGON ((20.59025 41.85541, 20.52295 42.21787...","[42.57742521451801, 20.895365849100475]","[Kosova, Kosowo, Koszovó, Republic of Kosovo, ...",country
175,Trinidad and Tobago,TT,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...","[10.427976277378917, -61.330418679071556]","[Republic of Trinidad and Tobago, Trinidad dan...",country


Let's filter out countries with an invalid alpha2 code

In [7]:
countries = countries[countries['iso_a2'] != "-99"]
countries = countries.reset_index(drop=True)
countries


,name,iso_a2,geometry,coordinates,aliases,level
0,Fiji,FJ,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...","[-17.31277520899394, 163.77640725005324]","[Fidji, Fidschi, Fidzsi-szigetek, Fidżi, Figi,...",country
1,United Republic of Tanzania,TZ,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...","[-6.237730725476361, 34.74766857623911]","[Tansania, Tanzania, Tanzanie, Tanzanya, Tanzá...",country
2,Western Sahara,EH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...","[24.24928060599045, -12.162183576752351]","[Batı Sahra, Nyugat-Szahara, Sahara Barat, Sah...",country
3,Canada,CA,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...","[57.62966510888876, -97.64486003045715]","[Canadá, Kanada, Καναδάς, Канада, קנדה, كندا, ...",country
4,United States of America,US,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...","[42.075007554015286, -107.5447286816333]","[Amerika Birleşik Devletleri, Amerika Serikat,...",country
...,...,...,...,...,...,...
167,North Macedonia,MK,"POLYGON ((22.38053 42.32026, 22.88137 41.9993,...","[41.60305470453288, 21.697076764970955]","[Bắc Macedonia, Kuzey Makedonya, Macedonia Pół...",country
168,Republic of Serbia,RS,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...","[44.21138805382481, 20.828267375143835]","[Serbia, Serbie, Serbien, Servië, Szerbia, Sér...",country
169,Montenegro,ME,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...","[42.78633660560693, 19.285947508559723]","[Czarnogóra, Karadağ, Montenegró, Monténégro, ...",country
170,Trinidad and Tobago,TT,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...","[10.427976277378917, -61.330418679071556]","[Republic of Trinidad and Tobago, Trinidad dan...",country


Let's inspect some sample data

In [8]:
countries[countries["name"] == "India"]

,name,iso_a2,geometry,coordinates,aliases,level
96,India,IN,"POLYGON ((97.32711 28.26158, 97.40256 27.88254...","[22.609142131655616, 79.57356544184115]","[Hindistan, Inde, Indie, Indien, Republic of I...",country


### Extract State Data
Next, let's import all the states in the countries we have extracted so far.

In [9]:
def build_state_aliases(states: gpd.GeoDataFrame):
    """Build flat list of alternative names for each state/province."""
    alt_name_columns = [
        col for col in states.columns
        if col.lower().startswith("name") or col in ["name_alt", "name_local", "name_ascii"]
    ]

    state_altnames = {}
    for _, row in states.iterrows():
        names = set()
        for col in alt_name_columns:
            v = row[col]
            if v not in [None, "", -99] and isinstance(v, str) and v != row["name"]:
                names.add(v)
        state_altnames[row["name"]] = sorted(names)
    return state_altnames


async def load_states():
    """Load all states/provinces worldwide with name, centroid coordinates, and aliases."""
    url = "http://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
    extracted_files = await download_and_extract(url)

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    states = gpd.read_file(shapefile)

    print("State data sample:")
    display(states.head())

    # Build aliases before dropping columns
    state_altnames = build_state_aliases(states)

    # Keep useful columns
    states = states[['name', "iso_a2",  'admin', 'geometry']]

   # Clean Alpha2 codes
    states['iso_a2'] = states['iso_a2'].apply(lambda x: x if str(x) != "-99" else None)

    # Project for accurate centroid
    states_proj = states.to_crs("ESRI:54034")  # Equal Earth projection
    centroids_proj = states_proj.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")
    states['coordinates'] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Map aliases
    states['aliases'] = states['name'].map(state_altnames)

    # Rename columns for clarity
    states = states.rename(columns={
        "admin": "country"
    })
    
    states["level"] = "state"

    return states

states = await load_states()
states

File already exists: data/ne_10m_admin_1_states_provinces.zip, skipping download.
Extraction complete.
State data sample:


,featurecla,scalerank,adm1_code,diss_me,iso_3166_2,wikipedia,iso_a2,adm0_sr,name,name_alt,...,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,FCLASS_TLC,geometry
0,Admin-1 states provinces,3,ARG-1309,1309,AR-E,None,AR,1,Entre Ríos,Entre-Rios,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-58.20011 -32.44713, -58.20012 -32.4..."
1,Admin-1 states provinces,6,URY-8,8,UY-PA,None,UY,1,Paysandú,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-58.20012 -32.4472, -58.20011 -32.44..."
2,Admin-1 states provinces,2,IDN-1185,1185,ID-KI,None,ID,5,Kalimantan Timur,Kaltim,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
3,Admin-1 states provinces,6,MYS-1186,1186,MY-12,None,MY,5,Sabah,North Borneo,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
4,Admin-1 states provinces,3,CHL-2694,2694,CL-AP,None,CL,1,Arica y Parinacota,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-69.51009 -17.50659, -69.50611 -17.5..."


,name,iso_a2,country,geometry,coordinates,aliases,level
0,Entre Ríos,AR,Argentina,"POLYGON ((-58.20011 -32.44713, -58.20012 -32.4...","[-32.02547013695879, -59.206689862401774]","[Entre Ríos eyaleti, Entre-Rios, Έντρε Ρίος, Е...",state
1,Paysandú,UY,Uruguay,"POLYGON ((-58.20012 -32.4472, -58.20011 -32.44...","[-32.082363780360275, -57.345107641978004]","[Departemen Paysandú, Paysandu Departmanı, dip...",state
2,Kalimantan Timur,ID,Indonesia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...","[1.352699778087046, 116.38394709926088]","[Borneo Wschodnie, Doğu Kalimantan, East Kalim...",state
3,Sabah,MY,Malaysia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...","[5.345010895470702, 117.13171580564357]","[North Borneo, Sabá, Σαμπάχ, Сабах, סבה, صباح,...",state
4,Arica y Parinacota,CL,Chile,"POLYGON ((-69.51009 -17.50659, -69.50611 -17.5...","[-18.485850925370954, -69.63872074583377]","[Arica e Parinacota, Arica et Parinacota, Aric...",state
...,...,...,...,...,...,...,...
4591,Isla,MT,Malta,"POLYGON ((14.51073 35.88896, 14.51537 35.89113...","[35.88612913557184, 14.514443273773145]","[Isla|Città Invicta, L-Isla, Senglea, Σενγκλέα...",state
4592,Cospicua,MT,Malta,"POLYGON ((14.52937 35.88156, 14.527 35.87772, ...","[35.87927143250959, 14.52043248348041]","[Bormla, Κοσπίκουα, Бормла, קוספיקאה, كوسبيكو,...",state
4593,Pietà,MT,Malta,"POLYGON ((14.49801 35.89251, 14.49616 35.8883,...","[35.88921922041928, 14.492441073423253]","[Πιετά, Пьета, Пієта, פיאטה, بييتا, پیئتا، مال...",state
4594,Msida,MT,Malta,"POLYGON ((14.48663 35.88636, 14.48231 35.88979...","[35.894557532084654, 14.48254329099217]","[Μσίντα, Мсида, Мсіда, מסידה, مسيدا, مسیدا, सी...",state


Let's inspect some sample state data

In [10]:
states[states["name"] == "Tamil Nadu"]

,name,iso_a2,country,geometry,coordinates,aliases,level
2242,Tamil Nadu,IN,India,"MULTIPOLYGON (((79.84734 10.80964, 79.84986 10...","[11.007117336162471, 78.40069620443494]","[Tamilnadu, Tamilnádu, Ταμίλ Ναντού, Тамилнад,...",state


### Extract City Data
Next, let's import all the cities in the states we have extracted so far.

In [11]:
CHUNK_SIZE = 100_000  # rows per chunk
PARQUET_DIR = pathlib.Path(f"{DATA_DIR}/geonames_parquet")
PARQUET_DIR.mkdir(exist_ok=True, parents=True)

def get_parquet_path(chunk_index):
    return PARQUET_DIR / f"geonames_chunk_{chunk_index}.parquet"

def build_aliases(row, alt_col: str = "NAMEALT"):
    names = set()
    for col in [alt_col]:
        v = row.get(col)
        if v and isinstance(v, str):
            # Split by comma and strip whitespace
            for name in v.split(","):
                name = name.strip()
                if name:
                    names.add(name)
    return sorted(names)


def process_chunk(chunk, chunk_index):
    """Process a single chunk and write to Parquet."""
    chunk['aliases'] = chunk.apply(build_aliases, axis=1)
    chunk['name'] = chunk['NAME']
    chunk['iso_a2'] = chunk['country_code']
    chunk['coordinates'] = chunk.apply(lambda r: [float(r['latitude']), float(r['longitude'])], axis=1)
    chunk['street_address'] = None
    chunk['postal_code'] = None
    chunk["address_region"] = None
    chunk["address_locality"] = None

    # make an explicit copy to avoid SettingWithCopyWarning
    address_chunk = chunk[[  
        'name', 'street_address', 'admin1_code', "admin2_code", "admin3_code", "admin4_code",
        "address_region", "address_locality", 'postal_code', 'iso_a2', 'coordinates', 'aliases'
    ]].copy()
    
    # clean iso a2 code
    address_chunk.loc[:, 'iso_a2'] = address_chunk['iso_a2'].apply(lambda x: x if str(x) != "-99" else None)

    parquet_path = get_parquet_path(chunk_index)
    address_chunk.to_parquet(parquet_path, index=False)
    print(f"Chunk {chunk_index} saved to {parquet_path}")
    return parquet_path

async def load_cities_localities_chunked_mp():
    geonames_url = "http://download.geonames.org/export/dump/allCountries.zip"
    extracted_files = await download_and_extract(geonames_url)
    geonames_file = next(pathlib.Path(f) for f in extracted_files if pathlib.Path(f).suffix == ".txt")

    columns = [
        "geonameid", "NAME", "NAMEASCII", "NAMEALT", "latitude", "longitude",
        "feature_class", "feature_code", "country_code", "cc2", "admin1_code",
        "admin2_code", "admin3_code", "admin4_code", "population", "elevation",
        "dem", "timezone", "modification_date"
    ]

    parquet_files = []

    # Read CSV in chunks
    chunks = pd.read_csv(
        geonames_file,
        sep="\t",
        header=None,
        names=columns,
        dtype=str,
        chunksize=CHUNK_SIZE
    )
    
    sample_chunks = pd.read_csv(
        geonames_file,
        sep="\t",
        header=None,
        names=columns,
        dtype=str,
        chunksize=CHUNK_SIZE
    )
    
    sample_df = next(sample_chunks)
    
    print("Localities / Cities data sample:")
    display(sample_df.head())  # Display first chunk sample

    with ProcessPoolExecutor() as executor:
        futures = []
        for i, chunk in enumerate(chunks):
            # Optional: filter populated places
            # feature classes:
            # A: country, state, region,...
            # H: stream, lake, ...
            # L: parks,area, ...
            # P: city, village,...
            # R: road, railroad 
            # S: spot, building, farm
            # T: mountain,hill,rock,... 
            # U: undersea
            # V: forest,heath,...
            chunk = chunk[chunk['feature_class'].isin(['P', 'L', 'A'])]
            futures.append(executor.submit(process_chunk, chunk, i + 1))

        for future in futures:
            parquet_files.append(future.result())

    print(f"All chunks saved. Total chunks: {len(parquet_files)}")
    return parquet_files

parquet_files = await load_cities_localities_chunked_mp()

File already exists: data/allCountries.zip, skipping download.
Extraction complete.
Localities / Cities data sample:


,geonameid,NAME,NAMEASCII,NAMEALT,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,population,elevation,dem,timezone,modification_date
0,2994701,Roc Meler,Roc Meler,"Roc Mele,Roc Meler,Roc Mélé",42.58765,1.7418,T,PK,AD,"AD,FR",02,NaN,NaN,NaN,0,2811,2348,Europe/Andorra,2023-10-03
1,3017832,Pic de les Abelletes,Pic de les Abelletes,"Pic de la Font-Negre,Pic de la Font-Nègre,Pic ...",42.52535,1.73343,T,PK,AD,FR,A9,66,663,66146,0,NaN,2411,Europe/Andorra,2014-11-05
2,3017833,Estany de les Abelletes,Estany de les Abelletes,"Estany de les Abelletes,Etang de Font-Negre,Ét...",42.52915,1.73362,H,LK,AD,FR,A9,NaN,NaN,NaN,0,NaN,2260,Europe/Andorra,2014-11-05
3,3023203,Port Vieux de la Coume d’Ose,Port Vieux de la Coume d'Ose,"Port Vieux de Coume d'Ose,Port Vieux de Coume ...",42.62568,1.61823,T,PASS,AD,NaN,00,NaN,NaN,NaN,0,NaN,2687,Europe/Andorra,2014-11-05
4,3029315,Port de la Cabanette,Port de la Cabanette,"Port de la Cabanette,Porteille de la Cabanette",42.6,1.73333,T,PASS,AD,"AD,FR",B3,09,091,09139,0,NaN,2379,Europe/Andorra,2014-11-05


Chunk 1 saved to data/geonames_parquet/geonames_chunk_1.parquet
Chunk 2 saved to data/geonames_parquet/geonames_chunk_2.parquet
Chunk 3 saved to data/geonames_parquet/geonames_chunk_3.parquet
Chunk 4 saved to data/geonames_parquet/geonames_chunk_4.parquet
Chunk 5 saved to data/geonames_parquet/geonames_chunk_5.parquet
Chunk 6 saved to data/geonames_parquet/geonames_chunk_6.parquet
Chunk 7 saved to data/geonames_parquet/geonames_chunk_7.parquet
Chunk 8 saved to data/geonames_parquet/geonames_chunk_8.parquet
Chunk 10 saved to data/geonames_parquet/geonames_chunk_10.parquet
Chunk 9 saved to data/geonames_parquet/geonames_chunk_9.parquet
Chunk 12 saved to data/geonames_parquet/geonames_chunk_12.parquet
Chunk 11 saved to data/geonames_parquet/geonames_chunk_11.parquet
Chunk 13 saved to data/geonames_parquet/geonames_chunk_13.parquet
Chunk 14 saved to data/geonames_parquet/geonames_chunk_14.parquet
Chunk 15 saved to data/geonames_parquet/geonames_chunk_15.parquet
Chunk 16 saved to data/geona

### Enrich city data with address regions and localities
the admin codes (`admin1_code`, `admin2_code` etc) are machine-readable codes in the earlier dataset. Let's convert them into human readable names, in the address_region and address_locality fields!

In [12]:
ADMIN1_URL = "http://download.geonames.org/export/dump/admin1CodesASCII.txt"
ADMIN2_URL = "http://download.geonames.org/export/dump/admin2Codes.txt"

async def download_admin_codes():
    """Download admin1 and admin2 codes into the data directory."""
    files = {}
    for url in [ADMIN1_URL, ADMIN2_URL]:
        extracted = await download_and_extract(url, extract_to=DATA_DIR)
        files[url] = extracted[0]  # single text file, not zipped
    return files


def load_admin_maps(admin1_file: str, admin2_file: str):
    """Load admin1 and admin2 mappings into dictionaries."""
    admin1_map, admin2_map = {}, {}

    # Admin1
    with open(admin1_file, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                code, name = parts[0], parts[1]
                admin1_map[code] = name

    # Admin2
    with open(admin2_file, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                code, name = parts[0], parts[1]
                admin2_map[code] = name

    return admin1_map, admin2_map


# 1. Download once
files = await download_admin_codes()

# 2. Load into dicts
admin1_map, admin2_map = load_admin_maps(
    files[ADMIN1_URL],
    files[ADMIN2_URL]
)

admin1_map, admin2_map

File already exists: data/admin1CodesASCII.txt, skipping download.
File already exists: data/admin2Codes.txt, skipping download.


({'AD.06': 'Sant Julià de Loria',
  'AD.05': 'Ordino',
  'AD.04': 'La Massana',
  'AD.03': 'Encamp',
  'AD.02': 'Canillo',
  'AD.07': 'Andorra la Vella',
  'AD.08': 'Escaldes-Engordany',
  'AE.07': 'Umm al Qaywayn',
  'AE.05': 'Raʼs al Khaymah',
  'AE.03': 'Dubai',
  'AE.06': 'Sharjah',
  'AE.04': 'Fujairah',
  'AE.02': 'Ajman',
  'AE.01': 'Abu Dhabi',
  'AF.28': 'Zabul',
  'AF.27': 'Maidan Wardak Province',
  'AF.26': 'Takhar',
  'AF.33': 'Sar-e Pol Province',
  'AF.32': 'Samangan',
  'AF.40': 'Parwan',
  'AF.29': 'Paktika',
  'AF.36': 'Paktia',
  'AF.39': 'Oruzgan',
  'AF.19': 'Nimroz',
  'AF.18': 'Nangarhar',
  'AF.17': 'Logar',
  'AF.35': 'Laghman',
  'AF.24': 'Kunduz',
  'AF.34': 'Kunar',
  'AF.14': 'Kapisa',
  'AF.23': 'Kandahar',
  'AF.13': 'Kabul',
  'AF.31': 'Jowzjan',
  'AF.11': 'Herat',
  'AF.10': 'Helmand',
  'AF.09': 'Ghowr',
  'AF.08': 'Ghazni',
  'AF.07': 'Faryab',
  'AF.06': 'Farah',
  'AF.05': 'Bamyan',
  'AF.30': 'Balkh',
  'AF.03': 'Baghlan',
  'AF.02': 'Badghis',
  

In [13]:
# Example of batch processing from Parquet files
def process_parquet_batches(parquet_files):
    """Yield DataFrames in batches from multiple Parquet files."""
    for chunk, file in enumerate(parquet_files):
        yield chunk + 1, pd.read_parquet(file, engine="pyarrow")

In [14]:
# 3. Use inside your pipeline
def resolve_address_region(row, admin1_map, admin2_map):
    parts = []

    # Admin1 lookup
    if row.get("iso_a2") and row.get("admin1_code"):
        key = f"{row['iso_a2']}.{row['admin1_code']}"
        if key in admin1_map:
            parts.append(admin1_map[key])

    return ", ".join(parts) if parts else None

def resolve_address_locality(row, admin1_map, admin2_map):
    parts = []

    # Admin2 lookup
    if row.get("iso_a2") and row.get("admin1_code") and row.get("admin2_code"):
        key = f"{row['iso_a2']}.{row['admin1_code']}.{row['admin2_code']}"
        if key in admin2_map:
            parts.append(admin2_map[key])

    # ignore Admin3/4 as they are just numbers. we need separate datasets to get human readable names for them

    return ", ".join(parts) if parts else None


def add_address_regions(df, admin1_map, admin2_map):
    df["address_region"] = df.apply(
        lambda r: resolve_address_region(r, admin1_map, admin2_map),
        axis=1
    )
    df["address_locality"] = df.apply(
        lambda r: resolve_address_locality(r, admin1_map, admin2_map),
        axis=1
    )
    return df

for chunk_index, batch_df in process_parquet_batches(parquet_files):
    # Do something with batch_df
    add_address_regions(batch_df, admin1_map, admin2_map)
    if chunk_index == 1:  # inspect first chunk
        display(batch_df.head())
    parquet_path = get_parquet_path(chunk_index)
    batch_df.to_parquet(parquet_path, index=False)
    print(f"Chunk {chunk_index} saved to {parquet_path}")

,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
0,Xixerella,None,04,None,None,None,La Massana,None,None,AD,"[42.55327, 1.48736]",[]
1,Visanseny,None,02,None,None,None,Canillo,None,None,AD,"[42.57488, 1.61616]","[Visanceny, Visanseny]"
2,Vila,None,03,None,None,None,Encamp,None,None,AD,"[42.53176, 1.56654]","[Casas Vila, Vila]"
3,Vial del Cardaire,None,04,None,None,None,La Massana,None,None,AD,"[42.56372, 1.50503]",[]
4,Pleta Vella,None,03,None,None,None,Encamp,None,None,AD,"[42.51143, 1.60976]",[]


Chunk 1 saved to data/geonames_parquet/geonames_chunk_1.parquet
Chunk 2 saved to data/geonames_parquet/geonames_chunk_2.parquet
Chunk 3 saved to data/geonames_parquet/geonames_chunk_3.parquet
Chunk 4 saved to data/geonames_parquet/geonames_chunk_4.parquet
Chunk 5 saved to data/geonames_parquet/geonames_chunk_5.parquet
Chunk 6 saved to data/geonames_parquet/geonames_chunk_6.parquet
Chunk 7 saved to data/geonames_parquet/geonames_chunk_7.parquet
Chunk 8 saved to data/geonames_parquet/geonames_chunk_8.parquet
Chunk 9 saved to data/geonames_parquet/geonames_chunk_9.parquet
Chunk 10 saved to data/geonames_parquet/geonames_chunk_10.parquet
Chunk 11 saved to data/geonames_parquet/geonames_chunk_11.parquet
Chunk 12 saved to data/geonames_parquet/geonames_chunk_12.parquet
Chunk 13 saved to data/geonames_parquet/geonames_chunk_13.parquet
Chunk 14 saved to data/geonames_parquet/geonames_chunk_14.parquet
Chunk 15 saved to data/geonames_parquet/geonames_chunk_15.parquet
Chunk 16 saved to data/geona

### Enrich city data with postal codes
We're almost done here, but we are missing the postal codes for the localities data. Let's add that now!

In [15]:
POSTAL_URL = "http://download.geonames.org/export/zip/allCountries.zip"

async def download_postal_codes():
    """Download and extract the global postal codes dataset."""
    extracted = await download_and_extract(POSTAL_URL, filename="all_countries_postal_codes.zip")
    return extracted[0]  # single .txt file inside


def load_postal_df(postal_file: str) -> pd.DataFrame:
    cols = [
        "country_code", "postal_code", "place_name",
        "admin1_name", "admin1_code",
        "admin2_name", "admin2_code",
        "admin3_name", "admin3_code",
        "latitude", "longitude", "accuracy"
    ]
    df = pd.read_csv(postal_file, sep="\t", header=None, names=cols, dtype=str)
    df["latitude"] = df["latitude"].astype(float)
    df["longitude"] = df["longitude"].astype(float)
    df = df.dropna(subset=["latitude", "longitude"])  # remove missing coords
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326"
    )
    return gdf

postal_file = await download_postal_codes()
postal_df = load_postal_df(postal_file)
print("Postal codes data sample:")
postal_df.head(20)

File already exists: data/all_countries_postal_codes.zip, skipping download.
Extraction complete.
Postal codes data sample:


,country_code,postal_code,place_name,admin1_name,admin1_code,admin2_name,admin2_code,admin3_name,admin3_code,latitude,longitude,accuracy,geometry
0,AD,AD100,Canillo,Canillo,02,NaN,NaN,NaN,NaN,42.5833,1.6667,6,POINT (1.6667 42.5833)
1,AD,AD200,Encamp,Encamp,03,NaN,NaN,NaN,NaN,42.5333,1.6333,6,POINT (1.6333 42.5333)
2,AD,AD400,La Massana,La Massana,04,NaN,NaN,NaN,NaN,42.5667,1.4833,6,POINT (1.4833 42.5667)
3,AD,AD300,Ordino,Ordino,05,NaN,NaN,NaN,NaN,42.6000,1.5500,6,POINT (1.55 42.6)
4,AD,AD600,Sant Julià de Lòria,Sant Julià de Lòria,06,NaN,NaN,NaN,NaN,42.4667,1.5000,6,POINT (1.5 42.4667)
5,AD,AD500,Andorra la Vella,Andorra la Vella,07,NaN,NaN,NaN,NaN,42.5000,1.5000,6,POINT (1.5 42.5)
6,AD,AD700,Escaldes-Engordany,Escaldes-Engordany,08,NaN,NaN,NaN,NaN,42.5000,1.5667,6,POINT (1.5667 42.5)
7,AE,28119 95762,Al Ras,NaN,03,NaN,100,Al Ras,112,25.2684,55.2931,6,POINT (55.2931 25.2684)
8,AE,28120 95556,Al Ras,NaN,03,NaN,100,Al Ras,112,25.2666,55.2932,6,POINT (55.2932 25.2666)
9,AE,28145 95641,Al Ras,NaN,03,NaN,100,Al Ras,112,25.2673,55.2934,6,POINT (55.2934 25.2673)


In [ ]:
from shapely.geometry import Point
from concurrent.futures import ProcessPoolExecutor, as_completed
import logging
logging.basicConfig(filename="process.log", level=logging.INFO)

PARQUET_GDF_DIR = pathlib.Path(f"{DATA_DIR}/geonames_gdf_parquet")
PARQUET_GDF_DIR.mkdir(exist_ok=True, parents=True)

def get_gdf_parquet_path(chunk_index):
    return PARQUET_GDF_DIR / f"geonames_chunk_{chunk_index}.parquet"

# --- Apply rules ---
def assign_postal(row):
    # if it's a high-level administrative unit → no postal code
    if (
        row["street_address"] is None
        and row["address_locality"] is None
    ):
        return None
    if row["name"] == row.get("address_locality"):
        return None
    return row["postal_code"]



def process_and_save(batch_index_df):
    chunk_index, batch_df = batch_index_df
    logging.info("loading dataframe...")
    locality_gdf = gpd.GeoDataFrame(
        batch_df,
        geometry=batch_df["coordinates"].apply(lambda x: Point(x[1], x[0])),
        crs="EPSG:4326"
    )
    locality_gdf = locality_gdf.to_crs(epsg=3857)
    postal_proj = postal_df.to_crs(epsg=3857)

    locality_with_postal = gpd.sjoin_nearest(
        locality_gdf,
        postal_proj[["postal_code", "geometry"]],
        how="left",
        distance_col="dist_m"
    )
    
    locality_with_postal["postal_code"] = locality_with_postal["postal_code_right"].combine_first(
        locality_with_postal["postal_code_left"]
    )

    # Drop the old columns
    locality_with_postal = locality_with_postal.drop(columns=["postal_code_left", "postal_code_right"])

    # locality_with_postal["postal_code"] = locality_with_postal.apply(assign_postal, axis=1)

    # --- Drop unnecessary columns ---
    locality_with_postal = locality_with_postal.drop(columns=["index_right", "dist_m"])
    locality_with_postal = locality_with_postal.set_geometry("geometry").to_crs("EPSG:4326")
    parquet_path = get_gdf_parquet_path(chunk_index)
    try:
        locality_with_postal.to_parquet(parquet_path, index=False)
        logging.info(f"Chunk {chunk_index} saved to {parquet_path}")
    except Exception as e:
        logging.info(f"Error saving chunk {chunk_index}: {e}")
    return parquet_path

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(process_and_save, batch): idx
        for idx, batch in enumerate(process_parquet_batches(parquet_files))
    }

    for future in as_completed(futures):
        idx = futures[future]
        try:
            result = future.result()  # forces execution & raises any errors
            print(f"✅ Chunk {idx} saved at {result}")
        except Exception as e:
            print(f"❌ Chunk {idx} failed: {e}")
            raise e

✅ Chunk 30 saved at data/geonames_gdf_parquet/geonames_chunk_31.parquet
✅ Chunk 126 saved at data/geonames_gdf_parquet/geonames_chunk_127.parquet
✅ Chunk 63 saved at data/geonames_gdf_parquet/geonames_chunk_64.parquet
✅ Chunk 104 saved at data/geonames_gdf_parquet/geonames_chunk_105.parquet
✅ Chunk 99 saved at data/geonames_gdf_parquet/geonames_chunk_100.parquet
✅ Chunk 91 saved at data/geonames_gdf_parquet/geonames_chunk_92.parquet
✅ Chunk 78 saved at data/geonames_gdf_parquet/geonames_chunk_79.parquet
✅ Chunk 122 saved at data/geonames_gdf_parquet/geonames_chunk_123.parquet
✅ Chunk 90 saved at data/geonames_gdf_parquet/geonames_chunk_91.parquet
✅ Chunk 65 saved at data/geonames_gdf_parquet/geonames_chunk_66.parquet
✅ Chunk 79 saved at data/geonames_gdf_parquet/geonames_chunk_80.parquet
✅ Chunk 1 saved at data/geonames_gdf_parquet/geonames_chunk_2.parquet
✅ Chunk 92 saved at data/geonames_gdf_parquet/geonames_chunk_93.parquet
✅ Chunk 26 saved at data/geonames_gdf_parquet/geonames_chun

Let's inspect some sample data now.

In [23]:
gdf_parquet_files = sorted(PARQUET_GDF_DIR.glob("geonames_chunk_*.parquet"))

for chunk_index, path in enumerate(gdf_parquet_files):
    gdf = gpd.read_parquet(path)
    display(gdf.head(3))

,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Xixerella,None,04,None,None,None,La Massana,None,None,AD,"[42.55327, 1.48736]",[],POINT (1.48736 42.55327),AD400
1,Visanseny,None,02,None,None,None,Canillo,None,None,AD,"[42.57488, 1.61616]","[Visanceny, Visanseny]",POINT (1.61616 42.57488),AD100
2,Vila,None,03,None,None,None,Encamp,None,None,AD,"[42.53176, 1.56654]","[Casas Vila, Vila]",POINT (1.56654 42.53176),AD700


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Fordlandia Park,None,16,1501006,None,None,Pará,Aveiro,None,BR,"[-3.8303, -55.49924]",[],POINT (-55.49924 -3.8303),68165-000
1,Praia de Taperapuan,None,05,2925303,None,None,Bahia,Porto Seguro,None,BR,"[-16.38671, -39.04005]",[],POINT (-39.04005 -16.38671),45807-000
2,Bracui,None,21,3300100,None,None,Rio de Janeiro,Angra dos Reis,None,BR,"[-22.94293, -44.38515]",[],POINT (-44.38515 -22.94293),23900-000


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Nedre Tappen,None,15,1864,None,None,Örebro,Ljusnarsbergs Kommun,None,SE,"[59.81667, 15.08333]",[],POINT (15.08333 59.81667),714 22
1,Nedre Tappen,None,15,1864,None,None,Örebro,Ljusnarsbergs Kommun,None,SE,"[59.81667, 15.08333]",[],POINT (15.08333 59.81667),714 35
2,Nedre Tappen,None,15,1864,None,None,Örebro,Ljusnarsbergs Kommun,None,SE,"[59.81667, 15.08333]",[],POINT (15.08333 59.81667),714 37


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Koyama,None,03,None,None,None,Southern Province,None,None,SL,"[8.13411, -11.37288]",[Koyama],POINT (-11.37288 8.13411),09813
1,Kpetema,None,01,None,None,None,Eastern Province,None,None,SL,"[8.11383, -11.31504]",[Kpetema],POINT (-11.31504 8.11383),09813
2,Dar-es-salaam,None,01,None,None,None,Eastern Province,None,None,SL,"[8.11446, -11.33592]",[Dar-es-salaam],POINT (-11.33592 8.11446),09813


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Mbovana,None,03,None,None,None,Manzini Region,None,None,SZ,"[-26.57747, 31.07024]",[],POINT (31.07024 -26.57747),2375
1,Mfukama,None,03,None,None,None,Manzini Region,None,None,SZ,"[-26.61427, 31.03748]",[],POINT (31.03748 -26.61427),2375
2,Dudusini,None,03,None,None,None,Manzini Region,None,None,SZ,"[-26.61715, 31.06952]",[],POINT (31.06952 -26.61715),2375


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Amphoe Na Tan,None,75,3430,None,None,Ubon Ratchathani,Amphoe Na Tan,None,TH,"[15.93485, 105.28454]","[Amphoe Na Tan, King Amphoe Na Tan, Na Tan, na...",POINT (105.28454 15.93485),34170
1,Suk Samran,None,59,8505,850502,None,Ranong province,Amphoe Suk Samran,None,TH,"[9.34458, 98.42903]","[Amphoe Suk Samran, King Amphoe Suk Samran, Su...",POINT (98.42903 9.34458),82150
2,Ban Takhun,None,60,8409,840901,None,Surat Thani,Amphoe Ban Takhun,None,TH,"[8.90364, 98.8845]","[Amphoe Ban Takhun, Ban Takhun, King Amphoe Ba...",POINT (98.8845 8.90364),84250


... (124 outputs left out)

,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Soplyaki,None,38,None,None,None,Krasnodar Krai,None,None,RU,"[46.76861, 39.71778]",[],POINT (39.71778 46.76861),352013
1,Daraganov,None,61,None,None,None,Rostov,None,None,RU,"[47.56839, 38.7766]","[Daraganov, Doroganov, Дараганов]",POINT (38.7766 47.56839),346991
2,Daraganov,None,61,None,None,None,Rostov,None,None,RU,"[47.56839, 38.7766]","[Daraganov, Doroganov, Дараганов]",POINT (38.7766 47.56839),346995


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Bukhta,None,64,None,None,None,Sakhalin Oblast,None,None,RU,"[45.38851, 148.48145]","[Bukhta, Sokitani, Sokiya, Бухта]",POINT (148.48145 45.38851),694535
1,Bryumka,None,92,None,None,None,Kamchatka,None,None,RU,"[54.52584, 155.81611]","[Brjumka, Bryumka, Брюмка]",POINT (155.81611 54.52584),684200
2,Bryanskoye,None,64,None,None,None,Sakhalin Oblast,None,None,RU,"[46.75, 142.31667]","[Bereyanskiy, Brjanskoe, Brjanskoje, Bryanskov...",POINT (142.31667 46.75),694036


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Tlyavgulovo,None,55,None,None,None,Orenburg Oblast,None,None,RU,"[51.4983, 57.2977]","[Tljavgulovo, Tlyavgulovo, Тлявгулово]",POINT (57.2977 51.4983),462243
1,Tlyavgulovo,None,55,None,None,None,Orenburg Oblast,None,None,RU,"[51.4983, 57.2977]","[Tljavgulovo, Tlyavgulovo, Тлявгулово]",POINT (57.2977 51.4983),462241
2,Tlyavgulovo,None,55,None,None,None,Orenburg Oblast,None,None,RU,"[51.4983, 57.2977]","[Tljavgulovo, Tlyavgulovo, Тлявгулово]",POINT (57.2977 51.4983),462244


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code_left,iso_a2,coordinates,aliases,geometry,postal_code_right
0,Rumāḩ,None,10,None,None,None,Riyadh Region,None,None,SA,"[25.56395, 47.16408]","[Ar Rimah, Ar Rimāḩ, Ar Rumah, Ar Rumāh, Rimah...",POINT (47.16408 25.56395),09874
1,Rumāḩ,None,10,None,None,None,Riyadh Region,None,None,SA,"[25.56395, 47.16408]","[Ar Rimah, Ar Rimāḩ, Ar Rumah, Ar Rumāh, Rimah...",POINT (47.16408 25.56395),09882
2,Rumāḩ,None,10,None,None,None,Riyadh Region,None,None,SA,"[25.56395, 47.16408]","[Ar Rimah, Ar Rimāḩ, Ar Rumah, Ar Rumāh, Rimah...",POINT (47.16408 25.56395),09881


In [18]:
def find_localities(names: list[str]) -> None:
    for chunk_index, batch_df in process_parquet_batches(parquet_files):
        # filter rows with missing postal_code        
        # further filter rows where 'name' matches the given name
        matched = batch_df[batch_df["name"].isin(names)]
        
        if not matched.empty:
            display(matched.head(20))
            
find_localities(["Alwarpet", "Vadapalani", "Chennai", "Tuticorin", "Coimbatore", "India", "Kerala"])


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
14436,India,None,07,None,None,None,North Bank,None,None,GM,"[13.56667, -15.75]",[]


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
16975,Chennai,None,25,603,None,None,Tamil Nadu,Chennai,None,IN,"[13.08784, 80.27847]","[Cenaj, Cenajo, Cenajus, Cenay, Cennai, Cennaj..."
19143,Kerala,None,25,None,None,None,Tamil Nadu,None,None,IN,"[11.0, 78.0]","[Chera, Kerala]"
24283,Coimbatore,None,25,632,5897,None,Tamil Nadu,Coimbatore,None,IN,"[11.00555, 76.96612]","[CJB, Coimbatore, Koimbator, Koimbatore, Koimb..."
24284,Coimbatore,None,25,632,None,None,Tamil Nadu,Coimbatore,None,IN,"[10.8, 77.09]","[CMT, Coimbatore, Coimbatore District, Coimbat..."
37209,Vadapalani,None,25,603,5982,None,Tamil Nadu,Chennai,None,IN,"[13.05111, 80.2125]","[Vadapalani, vatapalani, வடபழநி, വടപഴനി]"
37216,Alwarpet,None,25,603,5982,None,Tamil Nadu,Chennai,None,IN,"[13.03639, 80.25778]",[Alwarpet]
47328,Chennai,None,25,603,None,None,Tamil Nadu,Chennai,None,IN,"[13.08891, 80.26483]","[CHE, Chennai, Chennai District, Chennai distr..."


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
67567,India,None,35,435,3514,None,Madhya Pradesh,Ujjain,None,IN,"[23.01533, 75.36261]",[]


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
20298,Kerala,None,07,None,None,None,Batys Qazaqstan,None,None,KZ,"[49.14262, 51.12702]","[Kerala, Керала]"


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
17802,India,None,58,None,None,None,Transnistria,None,None,MD,"[47.04618, 29.3869]","[India, Indija, Indiya, Индия]"


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
20172,India,None,09,None,None,None,Zambezia Province,None,None,MZ,"[-16.69639, 36.91806]",[]


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
31654,India,None,12,None,None,None,San Vicente Department,None,None,SV,"[13.55, -88.81667]","[Caserio La India, Caserío La India, India, La..."


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
57794,India,None,TN,079,None,None,Tennessee,Henry County,None,US,"[36.33256, -88.2581]",[]


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
9603,India,None,MS,131,None,None,Mississippi,Stone County,None,US,"[30.81713, -89.11116]","[Inda, India]"


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
3096,India,None,TX,139,None,None,Texas,Ellis County,None,US,"[32.52514, -96.6161]","[India, Morgan]"


,name,street_address,admin1_code,admin2_code,admin3_code,admin4_code,address_region,address_locality,postal_code,iso_a2,coordinates,aliases
3343,India,None,PA,063,84184,None,Pennsylvania,Indiana County,None,US,"[40.37646, -79.11948]",[]
